In [ ]:
import pandas as pd
import subprocess
import shlex
import sys
import os
import matplotlib.pyplot as plt
import numpy as np
def execute(command):
    print (command)
    subprocess.call(shlex.split(command))

In [ ]:
counts_path = '../counts/'
ntc_path = '../ntc_counts/'
output_path = '../new_mageck/'
filtered_path = '../filtered_counts/'

for counts_file in os.listdir(counts_path):
    if counts_file.endswith('.txt'):
        df_merge=pd.read_table(counts_path+counts_file, index_col=0)
        sample = counts_file.split('.')[0]
        control_columns = [i for i in df_merge.columns.tolist() if 'ctrl' in i.lower()]
        controls = ','.join(control_columns)
        threshold = 100
        df_merge = df_merge[df_merge.loc[:,control_columns].mean(axis=1) > threshold]
        df_merge.to_csv(f'{filtered_path}{sample}_filtered.txt',sep = '\t')
        with open(ntc_path + sample+'_non-targeting.txt','w') as f:
            for i in [i for i in df_merge.index if 'non-' in i]:
                f.write(i + '\n')

        for time in ['2M','4M','10M','18M']:
            treatments = ','.join([i for i in df_merge.columns.tolist() if time in i.upper()])
            if len(treatments) > 0:
                screen_name = sample + '_' + time
                output = output_path + screen_name
                if screen_name not in os.listdir(output_path):
                    os.makedirs(output)

                execute("mageck test -k " 
            + f'{filtered_path}{sample}_filtered.txt' + " -t " 
            + treatments + " -c " + controls + " -n " + 
                        output + '/' + screen_name + " --pdf-report" + 
                        " --norm-method control --control-sgrna " + ntc_path + sample+'_non-targeting.txt')


In [ ]:
counts_path = '../counts/'
ntc_path = '../ntc_counts/'
output_path = '../roc_auc/mageck_output/'
filtered_path = '../filtered_counts/'

for counts_file in os.listdir(counts_path):
    if counts_file.endswith('.txt'):

        df_merge=pd.read_table(counts_path+counts_file, index_col=0)
        sample = counts_file.split('.')[0]
        control_columns = [i for i in df_merge.columns.tolist() if 'ctrl' in i.lower()]
        controls = ','.join(control_columns)
        treatment_columns =  [i for i in df_merge.columns.tolist()[1:] if not 'ctrl' in i.lower()]

        for treatment in treatment_columns:
            output = output_path + treatment
            screen_name = treatment
            if screen_name not in os.listdir(output_path):
                os.makedirs(output)
            try:
                execute("mageck test -k " 
            + f'{filtered_path}{sample}_filtered.txt' + " -t " 
            + treatment + " -c " + controls + " -n " + 
                        output + '/' + screen_name + " --pdf-report" + 
                        " --norm-method control --control-sgrna " + ntc_path + sample+'_non-targeting.txt')
            except:
                print ('error')